### ## Silver Notebook

## Lazy Evaluation

In [0]:
orders = spark.table("pfl_learning.bronze.orders")

completed_orders = (
    orders
    .filter(orders.order_status == "COMPLETED")
    .select(
        "order_id",
        "customer_id",
        "total_amount"
    )
)

completed_orders.count()

In [0]:
completed_orders.explain(True)

## Exercise 2: Jobs, Stages, Tasks

In [0]:
completed_orders.count()

## How to Access the Spark UI

Since this notebook runs on **Serverless (Spark Connect)** compute, `spark.sparkContext` is not available in the Python client. To inspect Jobs, Stages, and Tasks:

1. **From the notebook**: Click the compute name (top-right of the notebook) → this opens the compute details page.
2. On the compute details page, find the **Spark UI** tab/link.
3. In the Spark UI, click **Jobs** to see all Spark jobs triggered by actions like `count()`.

### What to Look For

| Spark UI Tab | What It Shows | Relation to `completed_orders.count()` |
| --- | --- | --- |
| **Jobs** | Each action (`count()`, `collect()`, etc.) creates one Spark Job | `completed_orders.count()` triggers **1 Job** |
| **Stages** | Each job is divided into stages (split by shuffles/exchanges) | Since `count()` only scans + filters (no shuffle), expect **1 Stage** |
| **Tasks** | Each stage is divided into tasks, one per partition | The number of tasks = number of file partitions in the `orders` table |

### Why `count()` Creates a Job

`count()` is an **action** — it triggers execution of the lazy DataFrame plan. The DAG (filter → select → count) compiled in Cell 4 (`explain(True)`) is materialized as:

```
Action: count()
  → Job 0
    → Stage 0
      → Task 0 (partition 1)
      → Task 1 (partition 2)
      → ... (one task per partition)
```

Run the cell below to see the event log summary after executing `count()` again:

In [0]:
# Re-run count() to trigger a fresh Spark Job
row_count = completed_orders.count()
print(f"Total completed orders: {row_count}")

# The physical plan shows how many partitions (tasks) will be scanned
print("\n--- Physical Plan (shows scan + partition count) ---")
completed_orders.explain("formatted")

In [0]:
%sql
SELECT 
    _metadata.file_path,
    _metadata.file_name,
    _metadata.file_size
FROM pfl_learning.silver.orders
LIMIT 10;

In [0]:
orders_50 = orders.repartition(50)

#orders_50.rdd.getNumPartitions()

In [0]:
orders.groupBy("shipping_city").count()   
orders.groupBy("shipping_city").count().explain(True)


In [0]:
%sql
select * from pfl_learning.silver.orders
limit 10;

# Exercise 6: Broadcast Join

In [0]:
customers= spark.table("pfl_learning.silver.customers")
orders=spark.table("pfl_learning.silver.orders")

from pyspark.sql.functions import broadcast
joined = (orders.join(broadcast(customers), "customer_id"))
display(joined)

In [0]:
from pyspark.sql.functions import broadcast, right
join1 = customers.join(orders, "customer_id","right")
display(join1)

#joined = orders.join(broadcast(customers), "customer_id")
#display(joined)

In [0]:

# spark.sql.autoBroadcastJoinThreshold is not available on Serverless (Spark Connect)
# Use DataFrame hints instead to control join strategy:
#   .hint("merge")        -> forces SortMergeJoin (no broadcast)
#   .hint("shuffle_hash") -> forces ShuffledHashJoin (no broadcast)
# Example: force a non-broadcast join without changing global config
join_no_broadcast = orders.join(customers.hint("merge"), "customer_id")
join_no_broadcast.explain(True)